In [1]:
from capsa.game import CapsaGame, display_cards
from capsa.players import HumanPlayer
from capsa.players.bots import RandomBot
from capsa.utils import generateCandidateMoves

players = [
    HumanPlayer("Player 1"),
    RandomBot("Player 2"),
    RandomBot("Player 3"),
    RandomBot("Player 4"),
]
game = CapsaGame(players)

game.state.lastPlayedCards = [
    # (3, 2), (6, 2), (8, 2), (11, 2), (12, 2)
    # 4♥, 7♥, 9♥, Q♥, K♥

    (10, 0), (10, 3),
    # J♦, J♠
]
game.state.cards = [
    4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
    4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4,
    4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4,
    4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
]

game.player_turn = 0
game.state.playerPassFlag = [False, False, False, False]

In [2]:
candidate_moves = generateCandidateMoves(1, game.state)

for i, move in enumerate(candidate_moves):
    print(f"{i}. ", end="")
    display_cards(move)

0. 
1. Q♣ Q♥ 


### Model Train

In [1]:
import mlflow
mlflow.set_tracking_uri("http://localhost:3000")
mlflow.set_experiment("Model Training")

<Experiment: artifact_location='mlflow-artifacts:/218620937189525767', creation_time=1748512566762, experiment_id='218620937189525767', last_update_time=1748512566762, lifecycle_stage='active', name='Model Training', tags={}>

In [2]:
from capsa import CapsaGame, logging
from capsa.players.bots import StateEvalBot
from tqdm import tqdm
from tempfile import TemporaryDirectory
import os
logging.disable_log()

players = [
    StateEvalBot("bot0", train=True),
    StateEvalBot("bot1", train=True),
    StateEvalBot("bot2", train=True),
    StateEvalBot("bot3", train=True),
]
game = CapsaGame(players)


with mlflow.start_run() as run:
    step = 0

    while True:
        try:
            game.start()

            for player in players:
                mlflow.log_metric(f"{player.name} Loss", player._loss, step=step)

            step += 1

            if step % 1000 == 0:
                for player in players:
                    with TemporaryDirectory() as tmpdir:
                        path = os.path.join(tmpdir, f"{player.name}-cp{step}.pt")
                        player.save_model(path)
                        mlflow.log_artifact(path)

        except KeyboardInterrupt:
            print("Training interrupted by user.")
            for player in players:
                with TemporaryDirectory() as tmpdir:
                    path = os.path.join(tmpdir, f"{player.name}.pt")
                    player.save_model(path)
                    mlflow.log_artifact(path)
            break

Training interrupted by user.
🏃 View run bouncy-toad-209 at: http://localhost:3000/#/experiments/218620937189525767/runs/3de74f7d71fd4ddba16e9db6b0c39e48
🧪 View experiment at: http://localhost:3000/#/experiments/218620937189525767


In [3]:
players[1].memory_y

tensor([  4.7205,   3.5800,   3.6242,   5.5344,   4.9856,   5.4352,   6.8372,
         13.0000,   5.5984,   6.6665,   6.5749,   7.3705,   8.2534,   9.6249,
         10.0987,  13.1917,  13.1032,  15.2135,  16.2740,  18.6063,  19.8538,
         23.2901,  25.4224,  27.1302,  27.4205,  36.5714,  -1.0666,  -1.0610,
         -2.9420,  -6.6142, -11.0547, -12.6146, -12.5597, -10.8431, -10.6124,
        -10.3444, -10.8829, -11.8684, -13.0394, -13.6089, -13.2764, -12.3941,
        -11.3081,  -8.0056,  -8.5098,  -7.2233,  -9.0983,  -6.9165,  -5.3974,
          0.2355,   0.4823,   0.1637,   0.6294,  -0.2455,   0.1939,  -1.4161,
          7.2528,  11.3781,  12.5590,  18.8395,  19.9572,  16.9221,  11.6563,
          7.9471,   6.3391,   4.2988,   3.1538,   0.8038,   2.4363,   4.9108,
          4.7786,   4.2219,   4.4738,   0.6981,   6.4391,   7.2140,   3.6119,
          9.7172,  10.2394,   7.9328,  27.0000,   2.2608,   2.8897,   2.9328,
          3.3678,   3.5056,   2.8280,   5.7232,   5.7045,   9.75

In [4]:
players[1].model(players[1].memory_X).flatten()

tensor([ 2.9630,  3.0034,  2.9626,  3.1187,  4.4971,  4.9288,  4.3507,  3.5825,
         2.8153,  5.9519,  4.8896,  6.8729,  8.4339,  9.3397,  8.9366, 12.1627,
         9.8685, 10.4898, 11.4007, 11.6748, 11.2706, 11.9852, 11.9665, 11.6652,
        11.2497, 10.5320, -0.2762, -0.2027, -0.2828, -0.1575, -0.2723, -0.2348,
        -0.0976,  0.6540,  1.0897,  0.6128,  0.8017,  0.4045,  0.1958,  0.2969,
         0.3249,  0.7494,  0.9380,  1.6214,  1.6350,  1.9024,  1.7669,  1.7070,
         1.8238,  0.6865,  1.9551,  2.2755,  3.0783,  3.0911,  2.0856,  1.7323,
         5.7761,  6.4136,  5.6618,  6.5123,  6.0991,  6.4144,  5.8063,  4.5897,
         3.1390,  3.4333,  3.8376,  5.3652,  0.2962,  0.6835,  0.9302,  0.6834,
         1.1476,  0.9776,  1.3826,  1.3862,  1.7845,  2.6309,  2.2929,  2.4867,
         2.3533,  2.6961,  2.2429,  2.2418,  2.4477,  3.5042,  3.3848,  4.9215,
         4.3785,  5.8159,  4.5948,  4.7734,  4.6673,  2.6664,  3.3381,  2.4919,
         2.7621,  3.5749,  5.0932,  4.28

In [5]:
for i, player in enumerate(players):
    player.save_model(f"./model_train/state_eval_s{i}.pt") 

### Compute Average Rank

In [5]:
from capsa import CapsaGame, logging
from capsa.players import HumanPlayer
from capsa.players.bots import RandomBot, StateEvalBot
logging.disable_log()

# players = [ StateEvalBot("Player 1"), StateEvalBot("Player 2"), StateEvalBot("Player 3"), StateEvalBot("Player 4") ] 
# players[0].load_model("./model_train/state_eval_s0.pt")
# players[1].load_model("./model_train/state_eval_s1.pt")
# players[2].load_model("./model_train/state_eval_s2.pt")
# players[3].load_model("./model_train/state_eval_s3.pt")

players = [ StateEvalBot("Player 1"), StateEvalBot("Player 2"), StateEvalBot("Player 3"), RandomBot("Player 4") ] 
players[0].load_model("./model_train/state_eval_group3.pt")
players[1].load_model("./model_train/state_eval_group2.pt")
players[2].load_model("./model_train/state_eval_s3.pt")

In [6]:
ranks = [0, 0, 0, 0]
num_games = 1000

game = CapsaGame(players)

for _ in range(num_games):
    game.start()

    for i, r in enumerate(game.ranks):
        ranks[i] += r + 1
        
[r / num_games for r in ranks]

[2.112, 2.017, 2.675, 3.196]

### Test Play Model

In [ ]:
from capsa import CapsaGame, logging
from capsa.players import HumanPlayer
from capsa.players.bots import RandomBot, StateEvalBot
logging.enable_log()

players = [ 
    StateEvalBot("Player 1", show_cards=True),
    StateEvalBot("Player 2", show_cards=True),
    StateEvalBot("Player 3", show_cards=True),
    StateEvalBot("Player 4", show_cards=True),
] 

# players[1].load_model("./model_train/state_eval_group3.pt")
# players[2].model = players[1].model
# players[3].model = players[1].model
# players[0].model = players[1].model

players[0].load_model("./model_train/state_eval_s0.pt")
players[1].load_model("./model_train/state_eval_s1.pt")
players[2].load_model("./model_train/state_eval_s2.pt")
players[3].load_model("./model_train/state_eval_s3.pt")

game = CapsaGame(players)
game.start()

[Player 3 Turn]

Cards in Hand:
A♦ 3♦ 5♦ 8♦ 10♦ J♦ A♣ 4♣ 7♣ 2♥ Q♥ K♥ 10♠ 

0. A♦  | Prediction: 24.237 | Move Reward: -8.362 | Rewards Breakdown: {'stopper_vs_liable': 0.6, 'hand_relative_worth': 1.4607843137254903, 'cards_in_hand': 0.07692307692307687, 'packs_broken': -10.5}
1. 3♦  | Prediction: 23.580 | Move Reward: -4.958 | Rewards Breakdown: {'stopper_vs_liable': 1.2, 'hand_relative_worth': 4.264705882352942, 'cards_in_hand': 0.07692307692307687, 'packs_broken': -10.5}
2. 5♦  | Prediction: 25.378 | Move Reward: -5.468 | Rewards Breakdown: {'stopper_vs_liable': 1.2, 'hand_relative_worth': 3.7549019607843164, 'cards_in_hand': 0.07692307692307687, 'packs_broken': -10.5}
3. 8♦  | Prediction: 18.594 | Move Reward: -3.233 | Rewards Breakdown: {'stopper_vs_liable': 1.2, 'hand_relative_worth': 2.9901960784313744, 'cards_in_hand': 0.07692307692307687, 'packs_broken': -7.5}
4. 10♦  | Prediction: 19.039 | Move Reward: -3.743 | Rewards Breakdown: {'stopper_vs_liable': 1.2, 'hand_relative_worth